# Laboratorio — Robot de entregas en un almacén

A partir de la **imagen**, construye el MDP y resuélvelo con **Value Iteration** y **Policy Iteration**.

![Mundo del ejercicio](https://drive.google.com/uc?export=view&id=1_sJaD57gHuiz1joEgl4B-u0aDy8jtMDo)



## Convención y notación

$$
s=(row,col)
$$

$$
T(s,a,s')=P(s'\mid s,a)
$$

$$
R(s)
$$

Para Value Iteration:

$$
V_{k+1}(s)
=
R(s)
+
\gamma
\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$

Para Policy Evaluation:

$$
V_{k+1}^{\pi}(s)
=
R(s)
+
\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Acciones

```python
UP    = (-1, 0)
DOWN  = ( 1, 0)
LEFT  = ( 0,-1)
RIGHT = ( 0, 1)
```



## Reglas del mundo

El grid tiene **5 filas × 6 columnas**.

### Estados especiales

A partir de la imagen identifica:

- `START`
- estanterías / paredes;
- zona de entrega `+10` (**terminal**);
- estación de carga `+2` (**terminal**);
- peligro mortal `-10` (**terminal**);
- peligros `-3` (**no terminales**);
- celdas de piso resbaloso.

### Recompensa

Usamos la convención del notebook de clase, es decir, **\(R(s)\)**:

- entrega: `+10`;
- carga: `+2`;
- peligro mortal: `-10`;
- peligro: `-3`;
- cualquier otro estado transitable: `-1` (costo por paso).

### Dinámica

La transición depende del **estado actual**:

**Piso normal**

$$
P(\text{dirección elegida})=0.90
$$

$$
P(\text{desviación izquierda})=0.05
$$

$$
P(\text{desviación derecha})=0.05
$$

**Piso resbaloso**

$$
P(\text{dirección elegida})=0.60
$$

$$
P(\text{desviación izquierda})=0.20
$$

$$
P(\text{desviación derecha})=0.20
$$

Si el movimiento sale del grid o golpea una estantería, el robot **permanece en el mismo estado**.

Usa:

$$
\gamma=0.9,\qquad \theta=10^{-4}
$$



## Parte 1 — Modela el MDP

Completa la clase `WarehouseMDP`.

La parte importante no es escribir muchas líneas de código: es traducir correctamente la imagen a:

- estados;
- acciones;
- recompensas;
- terminales;
- obstáculos;
- tipos de piso;
- función de transición.


In [ ]:
import numpy as np

class WarehouseMDP:
    def __init__(self):
        self.height = 5
        self.width = 6

        self.start = (4, 0)
        self.walls = {
            (0, 2), (0, 3),
            (1, 1), (1, 3),
            (2, 1), (2, 2),
            (3, 3), (3, 4),
        }
        self.slippery_states = {
            (1, 2), (2, 0), (3, 1), (3, 2), (2, 4)
        }

        self.terminal_states = {
            (0, 5): 10.0,
            (2, 5): 2.0,
            (4, 5): -10.0,
        }

        self.danger_states = {
            (1, 4): -3.0,
            (2, 3): -3.0,
            (4, 3): -3.0,
        }

        self.living_reward = -1.0
        self.gamma = 0.9

        self.actions = [
            (-1, 0),  # UP
            ( 1, 0),  # DOWN
            ( 0,-1),  # LEFT
            ( 0, 1),  # RIGHT
        ]

    def is_valid_state(self, state):
        r, c = state
        if not (0 <= r < self.height and 0 <= c < self.width):
            return False
        if (r, c) in self.walls:
            return False
        return True

    def states(self):
        return [
            (r, c)
            for r in range(self.height)
            for c in range(self.width)
            if self.is_valid_state((r, c))
        ]

    def is_terminal(self, state):
        return state in self.terminal_states

    def get_reward(self, state):
        if state in self.terminal_states:
            return self.terminal_states[state]
        if state in self.danger_states:
            return self.danger_states[state]
        if self.is_valid_state(state):
            return self.living_reward
        return 0.0

    def get_transition_probs(self, state, action):
        """
        Devuelve:
            [(next_state, probability), ...]

        Recuerda:
        - las probabilidades dependen de si 'state' es resbaloso;
        - si golpea pared/borde, next_state = state.
        """
        if self.is_terminal(state):
            return [(state, 1.0)]

        if action == (-1, 0):
            left = (0, -1)
            right = (0, 1)
        elif action == (1, 0):
            left = (0, 1)
            right = (0, -1)
        elif action == (0, -1):
            left = (1, 0)
            right = (-1, 0)
        else:
            left = (-1, 0)
            right = (1, 0)

        if state in self.slippery_states:
            outcomes = [(action, 0.60), (left, 0.20), (right, 0.20)]
        else:
            outcomes = [(action, 0.90), (left, 0.05), (right, 0.05)]

        result = {}
        for delta, probability in outcomes:
            next_state = (state[0] + delta[0], state[1] + delta[1])
            if not self.is_valid_state(next_state):
                next_state = state
            result[next_state] = result.get(next_state, 0.0) + probability

        return list(result.items())



### Validación mínima del modelo

Antes de implementar Bellman, valida primero el MDP.


In [ ]:
grid = WarehouseMDP()

S = grid.states()
print("Número de estados:", len(S))

# Cada distribución T(s,a,·) debe sumar 1.
for s in S:
    for a in grid.actions:
        transitions = grid.get_transition_probs(s, a)
        total = sum(p for _, p in transitions)
        assert abs(total - 1.0) < 1e-12

print("✓ Todas las distribuciones de transición suman 1.")



## Parte 2 — Value Iteration

Implementa:

$$
V_{k+1}(s)
=
R(s)+\gamma\max_a
\sum_{s'}T(s,a,s')V_k(s')
$$


In [ ]:
def expected_next_value(grid, state, action, V):
    value = 0.0
    for next_state, probability in grid.get_transition_probs(state, action):
        value += probability * V[next_state]
    return value


def value_iteration(grid, threshold=1e-4, max_iter=10_000):
    V = {s: 0.0 for s in grid.states()}

    for iteration in range(1, max_iter + 1):
        new_V = V.copy()
        delta = 0.0

        for s in grid.states():
            if grid.is_terminal(s):
                new_V[s] = grid.get_reward(s)
            else:
                best_value = -np.inf
                for action in grid.actions:
                    q = grid.get_reward(s) + grid.gamma * expected_next_value(grid, s, action, V)
                    if q > best_value:
                        best_value = q
                new_V[s] = best_value

            delta = max(delta, abs(new_V[s] - V[s]))

        V = new_V
        if delta < threshold:
            return V, iteration

    return V, max_iter


def extract_policy(grid, V):
    policy = {}
    for s in grid.states():
        if grid.is_terminal(s):
            continue

        best_action = None
        best_value = -np.inf
        for action in grid.actions:
            candidate = grid.get_reward(s) + grid.gamma * expected_next_value(grid, s, action, V)
            if candidate > best_value:
                best_value = candidate
                best_action = action

        policy[s] = best_action

    return policy



## Parte 3 — Policy Iteration

### Policy Evaluation

$$
V_{k+1}^{\pi}(s)
=
R(s)+\gamma
\sum_{s'}T(s,\pi(s),s')V_k^\pi(s')
$$

### Policy Improvement

$$
\pi_{\mathrm{new}}(s)
=
\arg\max_a
\sum_{s'}T(s,a,s')V^\pi(s')
$$

In [ ]:
def policy_evaluation(grid, policy, threshold=1e-4, max_iter=10_000):
    V = {s: grid.get_reward(s) for s in grid.states()}

    for _ in range(max_iter):
        new_V = V.copy()
        delta = 0.0

        for s in grid.states():
            if grid.is_terminal(s):
                new_V[s] = grid.get_reward(s)
            else:
                action = policy[s]
                value = grid.get_reward(s) + grid.gamma * expected_next_value(grid, s, action, V)
                new_V[s] = value

            delta = max(delta, abs(new_V[s] - V[s]))

        V = new_V
        if delta < threshold:
            break

    return V


def policy_improvement(grid, V, policy=None):
    if policy is None:
        policy = {}

    new_policy = {}
    for s in grid.states():
        if grid.is_terminal(s):
            continue

        current_action = policy.get(s, grid.actions[0])
        best_action = current_action
        best_value = grid.get_reward(s) + grid.gamma * expected_next_value(grid, s, current_action, V)

        for action in grid.actions:
            candidate = grid.get_reward(s) + grid.gamma * expected_next_value(grid, s, action, V)
            if candidate > best_value + 1e-12:
                best_value = candidate
                best_action = action
            elif abs(candidate - best_value) <= 1e-12 and action == current_action:
                # Mantener la política actual si hay un empate numérico.
                best_action = current_action

        new_policy[s] = best_action

    return new_policy


def policy_iteration(grid, threshold=1e-4, max_iter=100):
    policy = {s: grid.actions[0] for s in grid.states() if not grid.is_terminal(s)}
    history = []

    for _ in range(max_iter):
        V = policy_evaluation(grid, policy, threshold)
        new_policy = policy_improvement(grid, V, policy)
        history.append(len(new_policy))

        if new_policy == policy:
            return new_policy, V, history

        policy = new_policy

    return policy, V, history



## Parte 4 — Visualización y comparación


In [ ]:
ARROWS = {
    (-1, 0): "↑",
    ( 1, 0): "↓",
    ( 0,-1): "←",
    ( 0, 1): "→",
}

def print_values(grid, V):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)
            if s in grid.walls:
                row.append("  WALL  ")
            else:
                row.append(f"{V[s]:+7.3f}")
        print(" | ".join(row))


def print_policy(grid, policy):
    for r in range(grid.height):
        row = []
        for c in range(grid.width):
            s = (r, c)

            if s in grid.walls:
                row.append(" # ")
            elif grid.is_terminal(s):
                reward = grid.get_reward(s)
                row.append(f"{reward:+.0f}")
            else:
                row.append(f" {ARROWS[policy[s]]} ")

        print(" | ".join(row))


In [ ]:
# VALUE ITERATION
V_vi, n_vi = value_iteration(grid)
pi_vi = extract_policy(grid, V_vi)

print("=== VALUE ITERATION ===")
print("Iteraciones:", n_vi)
print("\nValores:")
print_values(grid, V_vi)
print("\nPolítica:")
print_policy(grid, pi_vi)


# POLICY ITERATION
pi_pi, V_pi, history = policy_iteration(grid)

print("\n=== POLICY ITERATION ===")
print("Historia:", history)
print("\nValores:")
print_values(grid, V_pi)
print("\nPolítica:")
print_policy(grid, pi_pi)

assert pi_vi == pi_pi
print("\n✓ Ambos algoritmos encontraron la misma política óptima.")



## Parte 5 — Interpreta la política

Antes de cambiar parámetros, responde:

1. Desde `START`, ¿el robot busca la **entrega +10** o prefiere la **estación de carga +2**?
2. ¿Por qué una recompensa menor podría ser óptima?
3. ¿En qué estados el piso resbaloso cambia la decisión?
4. ¿Qué papel cumple el costo por paso `-1`?
5. ¿Por qué \(T(s,a,s')\) ya no puede implementarse con las mismas probabilidades para todos los estados?

### Experimento A — Menos costo por paso

Cambia:

```python
living_reward = -0.1
```

Predice la política **antes de ejecutar**.

### Experimento B — Piso muy resbaloso

Cambia la probabilidad de movimiento deseado del piso resbaloso:

```python
0.60 → 0.40
```

y reparte el restante entre las dos desviaciones.

### Experimento C — Más paciencia

Cambia:

```python
gamma = 0.99
```

¿La política valora más la recompensa `+10` distante?

### Bonus

Encuentra aproximadamente el valor de `living_reward` a partir del cual la política desde `START` cambia entre:

- ir a carga `+2`;
- intentar llegar a entrega `+10`.


La decisión más razonable es priorizar la estación de carga si la ruta es más segura o si la entrega está lejos y exige más riesgo, aunque la recompensa final de +10 sea mayor. El robot no siempre va a la meta más grande: también compara la distancia, el riesgo y la incertidumbre del movimiento. El piso resbaloso cambia la elección porque aumenta la probabilidad de desviarse, así que la política suele evitar estados con mucho deslizamiento o elegir acciones que compensen ese error. El costo por paso -1 hace que el agente prefiera caminos cortos y eficientes, en lugar de prolongar la trayectoria solo por llegar a una recompensa más alta. La función T(s,a,s') depende del estado porque el mismo movimiento no tiene la misma probabilidad de éxito en un suelo normal que en uno resbaloso, ni al tocar una pared o un borde.